<a href="https://colab.research.google.com/github/jhhlim/LLMFundamentals/blob/cursor/hw-3-sentiment-topic-a051/Jason_Lim_hw_3b.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Class 3b — LLM Fundamentals (UCSC Extension)
# Homework 3b: Text Clustering and Topic Modeling

**Student:** Jason Lim  
**Submit to:** svagarwa@ucsc.edu  
**Filename:** `Jason_Lim_hw_3b.ipynb`

## Goal
Discover latent themes in unstructured free-text using the Class 3b / Hands-On LLM Ch. 5 pipeline:

1. **Embed** documents  
2. **Reduce** dimensions with **UMAP**  
3. **Cluster** with **HDBSCAN**  
4. Turn clusters into interpretable **topics** with **BERTopic** (c-TF-IDF keywords)

Also includes the class support warm-up: **Palmer Penguins + UMAP**.

## Dataset
**Hugging Face — `billingsmoore/text-clustering-example-data`**  
https://huggingface.co/datasets/billingsmoore/text-clustering-example-data

- 925 English sentences with a broad topic descriptor  
- Same dataset used in the class UMAP support notebook  
- Ideal size for Colab topic-modeling homework

## References
- Class support notebook: Read dataset / Penguins / UMAP  
- Class 3b / Hands-On LLM Ch. 5: embeddings → UMAP → HDBSCAN → BERTopic


### Install packages (Colab)

💡 **NOTE**: A GPU helps for sentence embeddings, but this homework dataset is small enough to run on CPU.


In [ ]:
%%capture
!pip install -q datasets umap-learn hdbscan bertopic sentence-transformers pandas matplotlib seaborn scikit-learn wordcloud


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")
%matplotlib inline


## Part A — Palmer Penguins UMAP warm-up (class support notebook)

Practice UMAP on a small numeric dataset before applying the same idea to text embeddings.


In [ ]:
penguins = pd.read_csv(
    "https://raw.githubusercontent.com/allisonhorst/palmerpenguins/"
    "c19a904462482430170bfe2c718775ddb7dbb885/inst/extdata/penguins.csv"
)
penguins = penguins.dropna()
print("Shape after dropna:", penguins.shape)
print(penguins.species.value_counts())
penguins.head()


In [ ]:
import umap

# Standard scaling: bill length/depth, flipper length, and body mass are on different scales.
penguin_data = penguins[
    ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
].values
scaled_penguin_data = StandardScaler().fit_transform(penguin_data)

reducer_2 = umap.UMAP(n_components=2, random_state=42)
embedding_2 = reducer_2.fit_transform(scaled_penguin_data)
print("2D embedding shape:", embedding_2.shape)

plt.figure(figsize=(7, 5))
plt.scatter(
    embedding_2[:, 0],
    embedding_2[:, 1],
    c=[
        sns.color_palette()[x]
        for x in penguins.species.map({"Adelie": 0, "Chinstrap": 1, "Gentoo": 2})
    ],
    s=25,
)
plt.gca().set_aspect("equal", "datalim")
plt.title("UMAP projection of the Penguin dataset", fontsize=14)
plt.show()


## Part B — Load the text clustering dataset

Class support notebook pattern:
```python
from datasets import load_dataset
dataset = load_dataset("billingsmoore/text-clustering-example-data")["train"]
```


In [ ]:
from datasets import load_dataset

dataset = load_dataset("billingsmoore/text-clustering-example-data")["train"]
df = dataset.to_pandas()
texts = df["text"].tolist()
true_topics = df["topic"].tolist()

print("Shape:", df.shape)
print("\nTopic counts:")
print(df["topic"].value_counts())
df.head()


## Part C — Common pipeline for text clustering (Class 3b / Ch. 5)

### 1) Embed documents with a sentence transformer


In [ ]:
from sentence_transformers import SentenceTransformer

# Compact, fast embedding model (works well on Colab CPU/GPU for ~1k docs)
embedding_model = SentenceTransformer("thenlper/gte-small")
embeddings = embedding_model.encode(texts, show_progress_bar=True)
print("Embedding shape:", embeddings.shape)


### 2) Reduce dimensionality with UMAP (for clustering)

In [ ]:
from umap import UMAP

# Reduce high-dim embeddings before density clustering (Ch. 5 pattern)
umap_model = UMAP(n_components=5, min_dist=0.0, metric="cosine", random_state=42)
reduced_embeddings = umap_model.fit_transform(embeddings)
print("Reduced embedding shape:", reduced_embeddings.shape)


### 3) Cluster with HDBSCAN

In [ ]:
from hdbscan import HDBSCAN

# Smaller min_cluster_size because this homework corpus is ~925 docs (not ArXiv-scale)
hdbscan_model = HDBSCAN(
    min_cluster_size=15,
    metric="euclidean",
    cluster_selection_method="eom",
)
clusters = hdbscan_model.fit_predict(reduced_embeddings)

n_clusters = len(set(clusters) - {-1})
n_outliers = int((clusters == -1).sum())
print("Clusters found:", n_clusters)
print("Outliers (label -1):", n_outliers)
print(pd.Series(clusters).value_counts().sort_index().head(20))


### Inspect a few documents from cluster 0

In [ ]:
cluster_id = 0
idxs = np.where(clusters == cluster_id)[0][:5]
print(f"Sample docs from cluster {cluster_id}:\n")
for i in idxs:
    print("-", texts[i], "| true topic:", true_topics[i])


### 2D UMAP visualization of clusters

In [ ]:
# Separate 2D reduction for plotting (same idea as the Ch. 5 notebook)
plot_embeddings = UMAP(
    n_components=2, min_dist=0.0, metric="cosine", random_state=42
).fit_transform(embeddings)

plot_df = pd.DataFrame(plot_embeddings, columns=["x", "y"])
plot_df["cluster"] = clusters.astype(str)
plot_df["true_topic"] = true_topics
plot_df["text"] = texts

clusters_df = plot_df[plot_df["cluster"] != "-1"]
outliers_df = plot_df[plot_df["cluster"] == "-1"]

plt.figure(figsize=(8, 6))
plt.scatter(outliers_df.x, outliers_df.y, alpha=0.15, s=12, c="grey", label="outlier")
plt.scatter(
    clusters_df.x,
    clusters_df.y,
    c=clusters_df.cluster.astype(int),
    alpha=0.75,
    s=18,
    cmap="tab20",
)
plt.axis("off")
plt.title("UMAP + HDBSCAN clusters on text embeddings")
plt.show()


## Part D — From clustering to topic modeling with BERTopic

BERTopic wraps the same embedding → UMAP → HDBSCAN stack and adds **c-TF-IDF** topic keywords.


In [ ]:
from bertopic import BERTopic

topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    verbose=True,
)
topics, probs = topic_model.fit_transform(texts, embeddings)

topic_info = topic_model.get_topic_info()
print(topic_info.head(12))
topic_info


### Top keywords for a few topics

In [ ]:
# Show top words for the largest non-outlier topics
non_outlier = topic_info[topic_info.Topic != -1].head(8)
for topic_num in non_outlier.Topic.tolist():
    words = topic_model.get_topic(topic_num)
    top = ", ".join([w for w, _ in words[:8]])
    print(f"Topic {topic_num}: {top}")


In [ ]:
# Compare discovered topics vs the dataset's provided topic labels (qualitative check)
cmp = pd.DataFrame({"bertopic": topics, "true_topic": true_topics})
print("Documents per BERTopic topic (top):")
print(cmp["bertopic"].value_counts().head(10))
print("\nCrosstab (BERTopic topic vs true label):")
ct = pd.crosstab(cmp["bertopic"], cmp["true_topic"])
ct.head(10)


### Topic visualizations

In [ ]:
# BERTopic charts (HTML widgets in Colab / Jupyter)
fig_bar = topic_model.visualize_barchart(top_n_topics=8)
fig_bar.show()

fig_docs = topic_model.visualize_documents(
    texts,
    reduced_embeddings=plot_embeddings,
    width=1000,
    hide_annotations=True,
)
fig_docs.update_layout(font=dict(size=14))
fig_docs.show()


## Part E — Bonus word cloud for one topic

(From the Ch. 5 bonus section.)


In [ ]:
from wordcloud import WordCloud

# Expand keywords for a nicer cloud
topic_model.update_topics(texts, top_n_words=50)

def create_wordcloud(model, topic):
    plt.figure(figsize=(10, 5))
    freqs = {word: value for word, value in model.get_topic(topic)}
    wc = WordCloud(background_color="white", max_words=200, width=1200, height=600)
    wc.generate_from_frequencies(freqs)
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title(f"Word cloud for topic {topic}")
    plt.show()

largest = int(topic_info[topic_info.Topic != -1].iloc[0].Topic)
create_wordcloud(topic_model, topic=largest)


## Learnings / analysis notes

1. **Penguins → text:** UMAP is the same idea on different inputs — numeric penguin measurements, then high-dimensional sentence embeddings.
2. **Pipeline:** Embeddings capture meaning → UMAP makes neighborhoods denser / lower-dim → HDBSCAN finds clusters without forcing a fixed `k`.
3. **Topic modeling vs clustering:** Clustering only groups docs; BERTopic adds c-TF-IDF keywords so each cluster becomes an interpretable topic.
4. **Dataset:** The HF clustering example set is small and labeled by broad topic, so we can visually check whether discovered clusters align with true themes.
5. **Product-review extension:** The same notebook pipeline can be pointed at UCI Amazon/Yelp review sentences for product-specific themes (battery, sound quality, service, etc.).
